In [27]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2

# 1. LOAD DATA
crabdata = pd.read_csv('CrabAgePrediction.csv') 



In [28]:
crabdata.head()

,Sex,Length,Diameter,Height,Weight,Shucked Weight,Viscera Weight,Shell Weight,Age
0,F,1.4375,1.1750,0.4125,24.635715,12.332033,5.584852,6.747181,9
1,M,0.8875,0.6500,0.2125,5.400580,2.296310,1.374951,1.559222,6
2,I,1.0375,0.7750,0.2500,7.952035,3.231843,1.601747,2.764076,6
3,F,1.1750,0.8875,0.2500,13.480187,4.748541,2.282135,5.244657,10
4,I,0.8875,0.6625,0.2125,6.903103,3.458639,1.488349,1.700970,6


In [29]:
crabdata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3893 entries, 0 to 3892
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Sex             3893 non-null   object 
 1   Length          3893 non-null   float64
 2   Diameter        3893 non-null   float64
 3   Height          3893 non-null   float64
 4   Weight          3893 non-null   float64
 5   Shucked Weight  3893 non-null   float64
 6   Viscera Weight  3893 non-null   float64
 7   Shell Weight    3893 non-null   float64
 8   Age             3893 non-null   int64  
dtypes: float64(7), int64(1), object(1)
memory usage: 273.9+ KB


In [30]:
#remove height=0
crabdata = crabdata[crabdata['Height'] > 0]

In [31]:
print(crabdata.shape)

(3891, 9)


In [32]:
# Factorize the 'Sex' column
crabdata['Sex'] = pd.factorize(crabdata['Sex'])[0]

# Change the numerical encoding to specific labels
sex_labels = {0: 'Female', 1: 'Male', 2: 'Indeterminate'}
crabdata['Sex'] = crabdata['Sex'].map(sex_labels)

# Show the first few rows of the DataFrame to verify the changes
print(crabdata.head())
crabdata.info()

             Sex  Length  Diameter  Height     Weight  Shucked Weight  \
0         Female  1.4375    1.1750  0.4125  24.635715       12.332033   
1           Male  0.8875    0.6500  0.2125   5.400580        2.296310   
2  Indeterminate  1.0375    0.7750  0.2500   7.952035        3.231843   
3         Female  1.1750    0.8875  0.2500  13.480187        4.748541   
4  Indeterminate  0.8875    0.6625  0.2125   6.903103        3.458639   

   Viscera Weight  Shell Weight  Age  
0        5.584852      6.747181    9  
1        1.374951      1.559222    6  
2        1.601747      2.764076    6  
3        2.282135      5.244657   10  
4        1.488349      1.700970    6  
<class 'pandas.core.frame.DataFrame'>
Index: 3891 entries, 0 to 3892
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Sex             3891 non-null   object 
 1   Length          3891 non-null   float64
 2   Diameter        3891 non-null   float64
 

In [33]:
#data on different scales. so we standardize
from sklearn.preprocessing import StandardScaler
#split data to features and label by making a copy of each
X=crabdata[["Sex","Length","Height","Weight","Diameter","Shucked Weight", "Viscera Weight", "Shell Weight"]].copy()
X['Shucked_Weight_Ratio'] = X['Shucked Weight'] / X['Weight']
X['Shell_Weight_Ratio'] = X['Shell Weight'] / X['Weight']
X['Viscera_Weight_Ratio'] = X['Viscera Weight'] / X['Weight']

X.drop(columns=['Weight', 'Shucked Weight', 'Viscera Weight', 'Shell Weight'], inplace=True)
Y=crabdata["Age"].copy()

In [34]:
Y.head()

0     9
1     6
2     6
3    10
4     6
Name: Age, dtype: int64

In [35]:
X.head()

,Sex,Length,Height,Diameter,Shucked_Weight_Ratio,Shell_Weight_Ratio,Viscera_Weight_Ratio
0,Female,1.4375,0.4125,1.1750,0.500575,0.273878,0.226697
1,Male,0.8875,0.2125,0.6500,0.425197,0.288714,0.254593
2,Indeterminate,1.0375,0.2500,0.7750,0.406417,0.347594,0.201426
3,Female,1.1750,0.2500,0.8875,0.352261,0.389064,0.169295
4,Indeterminate,0.8875,0.2125,0.6625,0.501027,0.246407,0.215606


In [36]:
X = X.drop(["Sex", "Shell_Weight_Ratio", "Viscera_Weight_Ratio"], axis=1)


In [37]:
X.head()

,Length,Height,Diameter,Shucked_Weight_Ratio
0,1.4375,0.4125,1.1750,0.500575
1,0.8875,0.2125,0.6500,0.425197
2,1.0375,0.2500,0.7750,0.406417
3,1.1750,0.2500,0.8875,0.352261
4,0.8875,0.2125,0.6625,0.501027


In [38]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


numerical_cols_to_scale = ["Length", "Diameter", "Height"]
numerical_cols_no_scale = ["Shucked_Weight_Ratio"]

# Define preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num_scale', StandardScaler(), numerical_cols_to_scale),
        ('num_no_scale', 'passthrough', numerical_cols_no_scale)
        
    ]
)

# Define the pipeline
pipeline = Pipeline(steps=[('preprocessor', preprocessor)])

# Fit and transform the data
X_processed = pipeline.fit_transform(X)
print(X_processed)

[[ 0.41953843  0.62039253  0.60126404  0.50057537]
 [-1.41179319 -1.49529346 -1.3091317   0.42519685]
 [-0.91233911 -0.9915587  -0.9509325   0.40641711]
 ...
 [-2.28583783 -2.25089561 -1.78673064  0.38028169]
 [-0.82909676 -0.9915587  -0.83153277  0.43561644]
 [-1.74476257 -1.64641389 -1.3091317   0.36933798]]


Spliit


In [39]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train_scaled, X_test_scaled, Y_train, Y_test = train_test_split(X_processed, Y,
                                                    train_size=0.8,
                                                    random_state=123)

# Make log transformation
Y_train_log=np.log1p(Y_train)
Y_test_transformed = np.log1p(Y_test)

In [40]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5]
}

rf_model_tuned = RandomForestRegressor()
grid_search_rf = GridSearchCV(rf_model_tuned, param_grid_rf, cv=5, scoring='neg_mean_squared_error')
grid_search_rf.fit(X_train_scaled, Y_train_log)

# Access the best parameters and best score
best_params_rf = grid_search_rf.best_params_
best_score_rf = grid_search_rf.best_score_

print("Best parameters for RandomForest:", best_params_rf)
print("Best score for RandomForest:", best_score_rf)

Best parameters for RandomForest: {'max_depth': 5, 'n_estimators': 200}
Best score for RandomForest: -0.030934830782507854


In [41]:
rf_model_best = RandomForestRegressor(**best_params_rf)
rf_model_best.fit(X_train_scaled, Y_train_log)

# Predictions on the training set
rf_train_predictions = rf_model_best.predict(X_train_scaled)

# Training MSE
rf_train_mse = mean_squared_error(Y_train_log, rf_train_predictions)

# Predictions on the test set
rf_test_predictions = rf_model_best.predict(X_test_scaled)

# Test MSE
rf_test_mse = mean_squared_error(Y_test_transformed, rf_test_predictions)

# RMSE for training predictions
rf_train_rmse = mean_squared_error(Y_train_log, rf_train_predictions, squared=False)

# RMSE for test predictions
rf_test_rmse = mean_squared_error(Y_test_transformed, rf_test_predictions, squared=False)

# MAE for training predictions
rf_train_mae = mean_absolute_error(Y_train_log, rf_train_predictions)

# MAE for test predictions
rf_test_mae = mean_absolute_error(Y_test_transformed, rf_test_predictions)

# R^2 for training predictions
rf_train_r2 = r2_score(Y_train_log, rf_train_predictions)

# R^2 for test predictions
rf_test_r2 = r2_score(Y_test_transformed, rf_test_predictions)

# Difference between train and test MSE
rf_mse_difference = rf_train_mse - rf_test_mse

# Calculate the difference between train and test MSE
rf_mse_difference = rf_train_mse - rf_test_mse

print("Random Forest Train MSE:", rf_train_mse)
print("Random Forest Test MSE:", rf_test_mse)
print("Random Forest Regression Train RMSE:", rf_train_rmse)
print("Random Forest Regression Test RMSE:", rf_test_rmse)
print("Random Forest Regression Train MAE:", rf_train_mae)
print("Random Forest Regression Test MAE:", rf_test_mae)
print("Random Forest Regression Train R^2:", rf_train_r2)
print("Random Forest Regression Test R^2:", rf_test_r2)
print("Random Forest Train-Test MSE Difference:", rf_mse_difference)

Random Forest Train MSE: 0.027387126046195583
Random Forest Test MSE: 0.029294175413516876
Random Forest Regression Train RMSE: 0.16549056180397595
Random Forest Regression Test RMSE: 0.17115541304182252
Random Forest Regression Train MAE: 0.1280363586843434
Random Forest Regression Test MAE: 0.12969685935873654
Random Forest Regression Train R^2: 0.6669659727752872
Random Forest Regression Test R^2: 0.6324593897853767
Random Forest Train-Test MSE Difference: -0.0019070493673212924
